# Semana 3: Tres sobres y una búsqueda del tesoro

**Introducción a la Ciencia de Datos** · Dudbil Olvasada Pabón Riaño · UNAB

Cuaderno guiado. Cada celda corresponde a un ejercicio de la guía de laboratorio; los resultados deben coincidir con sus puntos de control. Los datos se leen desde el repositorio del curso.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
BASE = "https://raw.githubusercontent.com/dudbilopr/Introduccion-a-la-ciencia-de-datos/main/Semanas/Datos/"

El mismo indicador del Banco Mundial (porcentaje de la población de Colombia que usa internet, datos de la UIT, CC BY 4.0) en los tres formatos que entrega su API.

## 1. CSV: abrir el sobre antes de leerlo

In [ ]:
import urllib.request
def leer_bytes(nombre):
    """Lee un archivo del repositorio del curso (en Colab, por internet)."""
    ruta = BASE + nombre
    return urllib.request.urlopen(ruta).read() if ruta.startswith("http") else open(ruta, "rb").read()

crudo = leer_bytes("banco_mundial_internet.csv").decode("utf-8-sig")
for i, linea in enumerate(crudo.splitlines()[:5], 1):
    print(i, linea[:90])

Las cuatro primeras líneas son metadatos. Sin `skiprows`, pandas falla o lee mal el encabezado.

In [ ]:
ancho = pd.read_csv(BASE + "banco_mundial_internet.csv", skiprows=4)
print("Forma ancha:", ancho.shape)
print("Columnas vacías por la coma final:", [c for c in ancho.columns if c.startswith("Unnamed")])
ancho = ancho.drop(columns=[c for c in ancho.columns if c.startswith("Unnamed")])
largo = ancho.melt(id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"], var_name="anio", value_name="pct")
largo["anio"] = largo["anio"].astype(int)
print("Forma larga:", largo.shape, " años con dato:", largo["pct"].notna().sum())
print(largo.dropna().head(3)[["anio", "pct"]])

## 2. JSON: aplanar lo anidado

In [ ]:
import json
J = json.loads(leer_bytes("banco_mundial_internet.json"))
print("Metadatos:", J[0])
print("Primer registro:", J[1][0])
dj = pd.json_normalize(J[1])[["date", "value", "indicator.id", "country.value"]]
dj["date"] = dj["date"].astype(int)
print(dj.head(3))

## 3. XML: etiquetas y espacio de nombres

In [ ]:
import io
txt_xml = leer_bytes("banco_mundial_internet.xml")
dx = pd.read_xml(io.BytesIO(txt_xml), xpath=".//wb:data", namespaces={"wb": "http://www.worldbank.org"}, parser="etree")
print(dx.columns.tolist())
print(dx[["date", "value"]].head(3))

## 4. ¿Dicen lo mismo los tres sobres?

In [ ]:
c = largo[(largo["anio"] >= 2000) & (largo["anio"] <= 2024)].sort_values("anio")["pct"].to_numpy()
j = dj.sort_values("date")["value"].to_numpy()
x = dx.sort_values("date")["value"].to_numpy()
print("CSV = JSON:", np.allclose(c, j), " CSV = XML:", np.allclose(c, x))
serie = dj.sort_values("date")
print("2000:", round(serie["value"].iloc[0], 2), "% · 2024:", round(serie["value"].iloc[-1], 2), "%")
print("Primer año con 50 % o más:", serie[serie["value"] >= 50]["date"].min())
plt.plot(serie["date"], serie["value"], marker="o"); plt.axhline(50, ls="--", c="gray")
plt.title("Colombia pasó la mitad de la población conectada en 2013"); plt.ylabel("% de la población"); plt.show()

## 5. Lo que pesa cada sobre

In [ ]:
bici = pd.read_csv(BASE + "bicicletas_dia.csv")
b_csv = len(bici.to_csv(index=False).encode())
b_json = len(bici.to_json(orient="records").encode())
b_xml = len(bici.to_xml(index=False, parser="etree").encode())
print("Bicicletas: CSV", b_csv, " JSON", b_json, " XML", b_xml)
print("JSON / CSV:", round(b_json / b_csv, 2), " XML / CSV:", round(b_xml / b_csv, 2))

## 6. Tipos de atributo (RAE 1.2)
Decida para cada columna si es nominal, ordinal, binaria o numérica. pandas solo ve números.

In [ ]:
print(bici[["season", "weathersit", "holiday", "cnt"]].dtypes)
print("Valores de season:", sorted(bici["season"].unique()), " de weathersit:", sorted(bici["weathersit"].unique()))
bici["season"] = bici["season"].astype("category")
bici["weathersit"] = pd.Categorical(bici["weathersit"], categories=[1, 2, 3, 4], ordered=True)
print(bici[["season", "weathersit"]].dtypes)

## 7. (Opcional) Pedirle los datos a la API en vivo
Requiere internet. Si el Banco Mundial actualizó la serie, los valores recientes pueden cambiar.

In [ ]:
url = "https://api.worldbank.org/v2/country/COL/indicator/IT.NET.USER.ZS?format=json&per_page=100&date=2000:2024"
try:
    vivo = json.loads(urllib.request.urlopen(url, timeout=30).read())
    print("Registros recibidos:", len(vivo[1]), " actualizado:", vivo[0]["lastupdated"])
except Exception as e:
    print("Sin conexión a la API:", e)

## Declaración de uso de IA

Si usó una herramienta de IA, escriba aquí: cuál, para qué, el prompt, la versión del modelo y qué verificó usted.